In [1]:
!curl -L -o ./the-babi-tasks-for-nlp-qa-system.zip https://www.kaggle.com/api/v1/datasets/download/roblexnana/the-babi-tasks-for-nlp-qa-system
!/usr/bin/unzip ./the-babi-tasks-for-nlp-qa-system.zip
!unlink ./the-babi-tasks-for-nlp-qa-system.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:--  0:00:01 --:--:--     0
100 16.6M  100 16.6M    0     0  3229k      0  0:00:05  0:00:05 --:--:-- 5704k
Archive:  ./the-babi-tasks-for-nlp-qa-system.zip
  inflating: tasks_1-20_v1-2/LICENSE.txt  
  inflating: tasks_1-20_v1-2/README.txt  
  inflating: tasks_1-20_v1-2/en-10k/qa10_indefinite-knowledge_test.txt  
  inflating: tasks_1-20_v1-2/en-10k/qa10_indefinite-knowledge_train.txt  
  inflating: tasks_1-20_v1-2/en-10k/qa11_basic-coreference_test.txt  
  inflating: tasks_1-20_v1-2/en-10k/qa11_basic-coreference_train.txt  
  inflating: tasks_1-20_v1-2/en-10k/qa12_conjunction_test.txt  
  inflating: tasks_1-20_v1-2/en-10k/qa12_conjunction_train.txt  
  inflating: tasks_1-20_v1-2/en-10k/qa13_compound-coreference_test.txt  
  inflating: tasks_1-20_v1-2/en-10k/qa13_compound-coreferenc

In [1]:
import sys, random, math
from collections import Counter
import numpy as np

In [2]:
f = open('./tasks_1-20_v1-2/en/qa1_single-supporting-fact_train.txt')
raw = f.readlines()
f.close()

In [3]:
sentences = list()
for line in raw[:1000]:
    sentences.append(line.lower().replace("\n", "").split(" ")[1:])

sentences[0:3]

[['mary', 'moved', 'to', 'the', 'bathroom.'],
 ['john', 'went', 'to', 'the', 'hallway.'],
 ['where', 'is', 'mary?', '\tbathroom\t1']]

In [4]:
vocab = set()
for sent in sentences:
    for word in sent:
        vocab.add(word)

vocab = list(vocab)
len(vocab)

82

In [5]:
word2index = {}
for i, word in enumerate(vocab):
    word2index[word] = i

len(word2index)

82

In [6]:
def sent2indices(sent):
    indices = list()
    for word in sent:
        indices.append(word2index[word])
    return indices

In [7]:
sent2indices(['where', 'is', 'mary'])

[78, 69, 63]

In [8]:
def softmax(v):
    e_v = np.exp(v - np.max(v))
    return e_v / e_v.sum(axis=0)

In [9]:
np.random.seed(1)

embed_size = 10

# Вложение слоев
word_embeddings = (np.random.rand(len(vocab), embed_size) - 0.5) * 0.1

# Recurrent. Рекурентная матрица (первоначально единичная)
W0 = np.eye(embed_size)

# Векторное представление для пустого предложения
empty_sentence_embedding = np.zeros(embed_size)

# Decoder. Выходные веса для прогнозирования векторного предстваления предложения
W1 = (np.random.rand(embed_size, len(vocab)) - 0.5) * 0.1

# Матрица поиска выходных весов (для функции потерь)
y_hots = np.eye(len(vocab))

In [10]:
def predict(sent):
    layers = list()
    layer = {}
    layer['hidden'] = empty_sentence_embedding
    layers.append(layer)

    loss = 0

    # прямое распространение
    for target_i in range(len(sent)):
        layer = {}

        # Прогнозирование следующего слова
        layer['pred'] = softmax(layers[-1]['hidden'].dot(W1))

        # Вычисление функции потерь для текущего слова
        # `sent[target_i]` получает актуальное слово, которое является числом, представляющим слово в словаре, затем мы получаем его предсказание в распределении вероятностей
        loss += -np.log(layer['pred'][sent[target_i]])

        # генерация следующего скрытого состояния
        layer['hidden'] = layers[-1]['hidden'].dot(W0) + word_embeddings[sent[target_i]]
        layers.append(layer)

    return layers, loss

In [11]:
for iter in range(30000):
    # forward propagation
    lr = .001
    # getting sentence indices w/o 1st word
    # why it leaves the first word of each sentence? -> to use it instead of "NaN" as first layer['hidden']
    sent = sent2indices(sentences[iter % len(sentences)][1:])
    layers, loss = predict(sent)  # predicts, returns hidden layer embeddings + loss

    # backpropagation
    for layer_i in reversed(range(len(layers))):  # loop over hidden states (layers)
        layer = layers[layer_i]  # get corresponding layer
        target = sent[layer_i - 1]  # get target word

        # If not the 1st layer
        if (layer_i > 0):  # if not first layer
            layer['output_delta'] = layer['pred'] - y_hots[target]  # delta
            new_hidden_delta = layer['output_delta'].dot(W1.transpose())  # gradient

            # If last layer, don't pull from a later one, because it doesn't exist
            # seems that for each hidden layer, its hidden delta is depedent upon the last decoding operation + next layer gradient
            if (layer_i == len(layers) - 1):
                layer['hidden_delta'] = new_hidden_delta
            else:
                layer['hidden_delta'] = new_hidden_delta + layers[layer_i + 1]['hidden_delta'].dot(W0.transpose())
        else:  # if the first layer
            layer['hidden_delta'] = layers[layer_i + 1]['hidden_delta'].dot(W0.transpose())

    # Update weights of NaN Embedding
    empty_sentence_embedding -= layers[0]['hidden_delta'] * lr / float(len(sent))
    for layer_i, layer in enumerate(layers[1:]):
        # update decoder
        W1 -= np.outer(layers[layer_i]["hidden"], layer['output_delta']) * lr / float(len(sent))
        embed_i = sent[layer_i]
        # update embeddings
        word_embeddings[embed_i] -= layers[layer_i]['hidden_delta'] * lr / float(len(sent))
        # update encoder
        W0 -= np.outer(layers[layer_i]['hidden'], layer['hidden_delta']) * lr / float(len(sent))

    if (iter % 1000 == 0):
        print("Perplexity :" + str(np.exp(loss / len(sent))))

Perplexity :82.1634014910116
Perplexity :81.96521048254336
Perplexity :81.70738837959199
Perplexity :81.27011385420913
Perplexity :80.41916051388954
Perplexity :78.57947647818713
Perplexity :73.84142073346827
Perplexity :54.963749820383086
Perplexity :27.237620444301065
Perplexity :20.066352649047083
Perplexity :18.249238120811977
Perplexity :16.398215231812227
Perplexity :13.644274897168488
Perplexity :10.173959735345512
Perplexity :7.720928814568061
Perplexity :6.487926012634615
Perplexity :5.698573308378984
Perplexity :5.181856256848575
Perplexity :4.880941775728069
Perplexity :4.694819447045347
Perplexity :4.584917762364899
Perplexity :4.521886013999044
Perplexity :4.4705934727011405
Perplexity :4.407616683540129
Perplexity :4.326175285196242
Perplexity :4.229296016067409
Perplexity :4.122582196075845
Perplexity :4.007027620426489
Perplexity :3.8955643587879316
Perplexity :3.8275723745899333


In [12]:
sent_index = 4
l, _ = predict(sent2indices(sentences[sent_index]))
print(sentences[sent_index])

['sandra', 'moved', 'to', 'the', 'garden.']


In [13]:
for i, each_layer in enumerate(l[1:-1]):
    input = sentences[sent_index][i]
    true = sentences[sent_index][i + 1]
    pred = vocab[each_layer['pred'].argmax()]
    print("Prev Input:" + input + (' ' * (12 - len(input))) + "True: " + true + (
        ' ' * (15 - len(true))) + "Pred: " + pred)

Prev Input:sandra      True: moved          Pred: is
Prev Input:moved       True: to             Pred: to
Prev Input:to          True: the            Pred: the
Prev Input:the         True: garden.        Pred: bedroom.
